# R.O.A.D. Historical HTR - Colab Training Pipeline

**Model:** Qwen2-VL-7B-Instruct  
**Optimized for:** Google Colab with A100 GPU  
**Expected Training Time:** 6-8 hours (A100), 12-15 hours (T4)  

---

**⚠️ IMPORTANT: Set Runtime to GPU (A100 recommended)**

1. Click `Runtime` → `Change runtime type`
2. Set `Hardware accelerator` → `GPU`
3. Set `GPU type` → `A100 40GB` (Colab Pro) or `A100 80GB` (Colab Pro+)
4. Click `Save`

---

**For T4 Users (Free/Pro tier):**  
Training is possible but slower. Reduce batch size in config if OOM:
```yaml
training:
  batch_size: 2  # down from 4
```

## 0. Verify GPU & Environment

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("\n❌ No GPU detected! Change runtime type to GPU.")

## 1. (Optional) Mount Google Drive

Mount Drive to save checkpoints persistently. Skip if you'll download them after training.

In [ ]:
# Uncomment to mount Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Set working directory in Drive (optional)
# %cd /content/drive/MyDrive/ROAD

print("✓ Skipping Drive mount (checkpoints will be in /content/)")

## 2. Clone Repository

In [ ]:
import os

# Update with your GitHub repo URL
REPO_URL = "YOUR_GITHUB_REPO_URL"  # e.g., "https://github.com/username/ROAD.git"
REPO_NAME = "ROAD"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
    print(f"✓ Cloned {REPO_NAME}")
else:
    print(f"✓ Repository {REPO_NAME} already exists")

%cd {REPO_NAME}

## 3. Download Image Dataset

Downloads 5,472 historical document images (~2-3 GB) from Google Cloud Storage.

In [ ]:
import os

IMAGE_URL = "https://storage.googleapis.com/road-handwriting/images.zip"
IMAGE_DIR = "dataset/images"

if not os.path.exists(IMAGE_DIR) or len(os.listdir(IMAGE_DIR)) < 5000:
    print("📥 Downloading images (~2-3 GB)...")
    !wget -q --show-progress {IMAGE_URL} -O images.zip
    
    print("📦 Extracting images...")
    !unzip -q images.zip -d dataset/
    !rm images.zip
    
    # Verify
    num_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
    print(f"✓ Downloaded {num_images} images")
else:
    num_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
    print(f"✓ Images already exist ({num_images} images)")

## 4. Install Dependencies

Installs Qwen2-VL, PEFT, Flash Attention, and training dependencies.

In [ ]:
%cd src/qwen2vl

print("📦 Installing dependencies (3-5 minutes)...")
!pip install -q -r requirements.txt

print("\n✓ Dependencies installed")

## 5. Verify Setup

In [ ]:
import torch
import pandas as pd
from pathlib import Path

print("=" * 70)
print("🔍 SETUP VERIFICATION")
print("=" * 70)

# GPU
print("\n[GPU]")
print(f"  Device: {torch.cuda.get_device_name(0)}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  BF16: {'✓' if torch.cuda.is_bf16_supported() else '✗'}")

# Dataset
print("\n[Dataset]")
train_csv = Path("../../dataset/Train.csv")
test_csv = Path("../../dataset/Test.csv")
image_dir = Path("../../dataset/images")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
num_images = len(list(image_dir.glob("*.jpg")))

print(f"  Train samples: {len(train_df)}")
print(f"  Test samples: {len(test_df)}")
print(f"  Images: {num_images}")

# Config
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

print("\n[Config]")
print(f"  Model: {cfg['model']['name']}")
print(f"  Batch size: {cfg['training']['batch_size']}")
print(f"  Effective batch: {cfg['training']['batch_size'] * cfg['training']['gradient_accumulation_steps']}")
print(f"  Epochs: {cfg['training']['epochs']}")
print(f"  LoRA rank: {cfg['training']['lora_r']}")
print(f"  Augmentation: {'✓' if cfg['augmentation']['enabled'] else '✗'}")

print("\n" + "=" * 70)
print("✓ Setup complete - ready to train")
print("=" * 70)

## 6. (Optional) Adjust Config for T4 GPU

If running on T4 (16GB VRAM) instead of A100, reduce batch size to avoid OOM.

In [ ]:
# Uncomment to auto-adjust for T4

# import torch
# import yaml

# gpu_name = torch.cuda.get_device_name(0)
# vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# if vram_gb < 30:  # T4 or smaller
#     print(f"🔧 Detected {gpu_name} ({vram_gb:.0f}GB) - adjusting config for lower VRAM")
    
#     with open("config.yaml") as f:
#         cfg = yaml.safe_load(f)
    
#     cfg['training']['batch_size'] = 2
#     cfg['training']['gradient_accumulation_steps'] = 8  # maintain effective batch=16
#     cfg['training']['lora_r'] = 32  # reduce rank
    
#     with open("config.yaml", "w") as f:
#         yaml.dump(cfg, f)
    
#     print("✓ Config adjusted: batch_size=2, lora_r=32")
# else:
#     print(f"✓ {gpu_name} detected - using default config")

print("Using default config (A100 optimized)")

## 7. Train Model 🚀

**Expected time:**
- A100 40GB: 6-8 hours
- A100 80GB: 6-8 hours  
- T4 16GB: 12-15 hours (with adjusted batch size)

**Memory usage:**
- A100: ~45-55 GB
- T4: ~14-15 GB (with batch_size=2, lora_r=32)

The training will:
- Load Qwen2-VL-7B-Instruct (~15 GB)
- Apply LoRA fine-tuning (rank 64)
- Use image augmentation (blur, noise, contrast, rotation)
- Evaluate every 100 steps
- Save best model based on eval loss

**Note:** Colab may disconnect after 12 hours. For long training, consider:
1. Mounting Drive to save checkpoints
2. Using smaller LoRA rank or fewer epochs
3. Enabling `save_steps` to save intermediate checkpoints

In [ ]:
import time
start = time.time()

print("🚀 Starting training...\n")
!python train.py

elapsed = (time.time() - start) / 3600
print(f"\n✓ Training completed in {elapsed:.1f} hours")

## 8. Generate Submission

Run inference on test set using the best checkpoint.

In [ ]:
print("🔮 Running inference on test set...\n")
!python inference.py

print("\n✓ Submission generated: ../../submission.csv")

## 9. Verify Submission

In [ ]:
import pandas as pd

submission = pd.read_csv("../../submission.csv")

print("=" * 70)
print("📊 SUBMISSION PREVIEW")
print("=" * 70)
print(submission.head(10))

print("\n" + "=" * 70)
print("📈 SUBMISSION STATS")
print("=" * 70)
print(f"Total predictions: {len(submission)}")
print(f"Empty predictions: {(submission['Target'] == '').sum()}")
print(f"Avg text length: {submission['Target'].str.len().mean():.1f} chars")
print(f"Min text length: {submission['Target'].str.len().min():.0f} chars")
print(f"Max text length: {submission['Target'].str.len().max():.0f} chars")

print("\n✓ Submission ready: ../../submission.csv")
print("=" * 70)

## 10. Download Files

Download submission and checkpoints to your local machine.

In [ ]:
from google.colab import files

# Download submission
print("Downloading submission.csv...")
files.download("../../submission.csv")

# Optional: Download best checkpoint (large file)
# print("\nPacking best checkpoint...")
# !tar -czf checkpoint_best.tar.gz outputs/qwen2vl-7b-run1/best/
# files.download("checkpoint_best.tar.gz")

print("\n✓ Download complete")

## 11. (Optional) Sample Predictions Visualization

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Sample 3 random predictions
samples = submission.sample(3)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

for ax, (_, row) in zip(axes, samples.iterrows()):
    img_path = f"../../dataset/images/{row['ID']}.jpg"
    img = Image.open(img_path)
    
    ax.imshow(img)
    ax.set_title(f"Prediction: {row['Target'][:100]}...", fontsize=9, pad=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---

## 🎯 Next Steps for Improvement

### 1. Error Analysis
- Review predictions on validation set
- Identify common error patterns (missing words, wrong letters)
- Adjust augmentation strategy based on errors

### 2. Hyperparameter Tuning
- **Increase epochs:** Try 7-10 epochs
- **Learning rate:** Experiment with 1e-5 to 3e-5
- **LoRA rank:** Try 32, 64, 128 (higher = more capacity)
- **Augmentation:** Adjust probabilities based on error patterns

### 3. Ensemble Strategy
- Train multiple models with different seeds
- Train TrOCR-large for architectural diversity
- Combine predictions with weighted voting
- Test-time augmentation (3-5 passes per image)

### 4. Post-processing
- Historical spelling correction
- Language model filtering (GPT for common phrases)
- Common pattern detection (dates, names, etc.)

### 5. Advanced Techniques
- Curriculum learning (train on easy samples first)
- Focal loss for hard examples
- Self-training on test set pseudo-labels

---

## 📊 Expected Performance

| Configuration | WER | CER | Combined Score |
|--------------|-----|-----|----------------|
| Baseline (no aug) | 15-20% | 5-8% | 10-14% |
| + Augmentation | 12-15% | 4-6% | 8-10.5% |
| + Ensemble (2-3 models) | 10-12% | 3-5% | 6.5-8.5% |
| + Post-processing | 8-10% | 2-4% | 5-7% |

**Target for top 10%:** Combined score < 8%  
**Target for top 3%:** Combined score < 6%

---

Good luck! 🚀